## 03 - pytorch - Tensors & Autograd

Using PyTorch Tensors and autograd to implement our fitting sine wave with third order polynomial example; now we no longer need to manually implement the backward pass through the network

In [1]:
import torch
import math

In [2]:
dtype = torch.float
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")
torch.set_default_device(device)

Using device: cuda


In [3]:
# create random input & output - without the need for gradients
x = torch.linspace(-1, 1, 2000, dtype=dtype)
y = torch.exp(x) # Taylor expansion - 1 + x + (1/2) * x ** 2 + (1/3!) * x ** 3 + ... 

# create random tensors for weights, for 3rd order polynomial we need 4 weights
# y = a + bx + cx**2 + dx**3 & for each we will need to compute the gradients
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)

initial_loss = -1
learning_rate = 1e-6

In [5]:
for t in range(2000):
    y_pred = a + b*x + c*x**2 + d*x**3

    loss = (y_pred - y).pow(2).sum()

    # Calculare initial loss, so we can report loss relative to it
    if t==0:
        initial_loss=loss.item()

    if t % 100 == 99:
        print(f'Iteration t = {t:4d}  loss(t)/loss(0) = {round(loss.item()/initial_loss, 6):10.6f}  a = {a.item():10.6f}  b = {b.item():10.6f}  c = {c.item():10.6f}  d = {d.item():10.6f}')

    # use autograd
    loss.backward()

    # update weights
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad

        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

Iteration t =   99  loss(t)/loss(0) =   0.600850  a =   0.366367  b =  -1.725949  c =  -0.080552  d =   0.263545
Iteration t =  199  loss(t)/loss(0) =   0.378672  a =   0.635597  b =  -1.399583  c =   0.029206  d =   0.457848
Iteration t =  299  loss(t)/loss(0) =   0.248443  a =   0.806081  b =  -1.127275  c =   0.102790  d =   0.618062
Iteration t =  399  loss(t)/loss(0) =   0.167878  a =   0.913604  b =  -0.899932  c =   0.153185  d =   0.749934
Iteration t =  499  loss(t)/loss(0) =   0.115990  a =   0.980996  b =  -0.709991  c =   0.188676  d =   0.858243
Iteration t =  599  loss(t)/loss(0) =   0.081632  a =   1.022822  b =  -0.551162  c =   0.214554  d =   0.946964
Iteration t =  699  loss(t)/loss(0) =   0.058461  a =   1.048372  b =  -0.418214  c =   0.234197  d =   1.019404
Iteration t =  799  loss(t)/loss(0) =   0.042646  a =   1.063573  b =  -0.306797  c =   0.249764  d =   1.078310
Iteration t =  899  loss(t)/loss(0) =   0.031765  a =   1.072206  b =  -0.213292  c =   0.262636